# Recompra a 60 días — The Look eCommerce
**Caso práctico Machine Learning Engineer · Tais Lisette Arrunátegui Matos**

**Pregunta de negocio:** ¿qué clientes volverán a comprar en los 60 días siguientes a su primera compra?
**Recompra** = una **nueva orden** (no ítems de la misma orden) en ese período.

| # | Sección | Qué resuelve |
|---|---|---|
| 0 | Configuración | Parámetros únicos de toda la ejecución (fecha de corte, ventana, estados) |
| 1 | Calidad de datos fuente | Duplicados, nulos, huérfanos, estados y fechas, en SQL |
| 2 | Tabla base (SQL) | 1 fila por cliente, etiqueta sin fuga, clientes sin 60 días de historia |
| 3 | Análisis exploratorio | Tasa de recompra, tiempo entre compras, segmentos |
| 4 | Modelado y evaluación | Split temporal, referencia vs. modelos, PR AUC, lift |
| 5 | Predicciones | Scoring de clientes recientes, guardado idempotente |
| 6 | Ejecución de punta a punta | `ejecutar_pipeline(CONFIG)` sin pasos manuales + prueba de re-ejecución |

**Cómo ejecutar:** *Kernel → Restart & Run All* (local) o *Entorno de ejecución → Ejecutar todo* (Colab).
Solo hay que revisar la celda de configuración.

## 0. Configuración

In [ ]:
import datetime as dt

CONFIG = {
    # Origen y destino en BigQuery
    "PROYECTO":            "bussiness-case-tyc",
    "DATASET_ORIGEN":      "bussiness_case_ecommerce",
    "DATASET_DESTINO":     "tais_arrunategui",          # dataset propio de la candidata
    # Definición del problema
    # Exclusiva. La última orden observada es del 14/02/2025 → corte 15/02/2025 usa toda la historia.
    # None = 1er día del mes de la última orden (cálculo automático).
    "FECHA_CORTE":         dt.date(2025, 2, 15),
    "VENTANA_DIAS":        60,
    "ESTADOS_EXCLUIDOS":   ["Cancelled"],   # una orden cancelada no es una compra
    # Modelado
    "FRACCION_TEST":       0.20,     # cohortes más recientes reservadas para evaluación
    "CAPACIDAD_CAMPANA":   0.20,     # % de clientes que el negocio puede contactar
    "SEMILLA":             42,
    # Salidas
    "GUARDAR_EN_BQ":       True,     # escribe tabla base y predicciones en el dataset destino
    "REPO_URL":            "https://github.com/<tu-usuario>/recompra-60d-tais-arrunategui.git",
}
CONFIG

In [ ]:
import json, math, os, subprocess, sys, warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60, "display.width", 180)

# En Colab: autenticar y clonar el repo para tener los archivos .sql
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    if not Path("recompra-60d-tais-arrunategui").exists():
        subprocess.run(["git", "clone", CONFIG["REPO_URL"]], check=True)
    os.chdir("recompra-60d-tais-arrunategui")

# Raíz del repo: funciona tanto desde notebooks/ como desde la raíz
RAIZ = Path.cwd() if (Path.cwd() / "sql").exists() else Path.cwd().parent
DIR_SQL, DIR_SALIDAS, DIR_MODELOS = RAIZ / "sql", RAIZ / "outputs", RAIZ / "models"
DIR_FIGURAS = DIR_SALIDAS / "figuras"
for d in (DIR_SALIDAS, DIR_MODELOS, DIR_FIGURAS):
    d.mkdir(parents=True, exist_ok=True)
print("Raíz del repo:", RAIZ)

### Acceso a BigQuery
Toda la interacción con BigQuery está en esta celda. Las consultas usan **parámetros de consulta**
(`@fecha_corte`, `@ventana_dias`, `@estados_excluidos`), no concatenación de texto.
Las escrituras son **idempotentes**: la tabla base se reemplaza (`WRITE_TRUNCATE`) y las predicciones
reemplazan solo la partición de su fecha de corte.

In [ ]:
from google.api_core.exceptions import NotFound
from google.cloud import bigquery

client = bigquery.Client(project=CONFIG["PROYECTO"])


def _parametro(nombre, valor):
    """Convierte un valor de Python en un parámetro tipado de BigQuery."""
    if isinstance(valor, (list, tuple)):
        return bigquery.ArrayQueryParameter(nombre, "STRING", list(valor))
    if isinstance(valor, dt.date):
        return bigquery.ScalarQueryParameter(nombre, "DATE", valor)
    if isinstance(valor, (int, np.integer)):
        return bigquery.ScalarQueryParameter(nombre, "INT64", int(valor))
    if isinstance(valor, float):
        return bigquery.ScalarQueryParameter(nombre, "FLOAT64", valor)
    return bigquery.ScalarQueryParameter(nombre, "STRING", str(valor))


def consultar(sql, params=None, destino=None):
    """Ejecuta una consulta. Si se indica `destino`, además la escribe con WRITE_TRUNCATE."""
    cfg = bigquery.QueryJobConfig(
        query_parameters=[_parametro(k, v) for k, v in (params or {}).items()])
    if destino:
        cfg.destination = destino
        cfg.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()
    gb = (job.total_bytes_processed or 0) / 1e9
    print(f"  BigQuery: {gb:.3f} GB procesados")
    return df


def asegurar_dataset_destino(cfg):
    """Crea el dataset de la candidata si no existe, en la misma región que el origen."""
    origen = client.get_dataset(f'{cfg["PROYECTO"]}.{cfg["DATASET_ORIGEN"]}')
    ds = bigquery.Dataset(f'{cfg["PROYECTO"]}.{cfg["DATASET_DESTINO"]}')
    ds.location = origen.location
    client.create_dataset(ds, exists_ok=True)


def reemplazar_particion(df, tabla, fecha_corte):
    """Guarda `df` en `tabla` (particionada por fecha_corte) reemplazando SOLO esa fecha.
    Re-ejecutar con la misma fecha de corte no duplica filas."""
    try:
        client.get_table(tabla)
        consultar(f"DELETE FROM `{tabla}` WHERE fecha_corte = @fc", {"fc": fecha_corte})
    except NotFound:  # primera ejecución: la tabla la crea la carga
        pass
    job_cfg = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
        time_partitioning=bigquery.TimePartitioning(field="fecha_corte"),
    )
    client.load_table_from_dataframe(df, tabla, job_config=job_cfg).result()


def contar_filas(tabla, fecha_corte):
    return int(consultar(f"SELECT COUNT(*) AS n FROM `{tabla}` WHERE fecha_corte = @fc",
                         {"fc": fecha_corte}).n.iloc[0])

In [ ]:
def nombre_tabla(cfg, tabla, destino=False):
    ds = cfg["DATASET_DESTINO"] if destino else cfg["DATASET_ORIGEN"]
    return f'{cfg["PROYECTO"]}.{ds}.{tabla}'


def cargar_sql(nombre, cfg):
    """Lee un archivo de sql/ y ajusta proyecto y datasets según CONFIG
    (los archivos traen los nombres reales para poder ejecutarse en la consola)."""
    sql = (DIR_SQL / nombre).read_text(encoding="utf-8")
    sql = sql.replace("bussiness-case-tyc.bussiness_case_ecommerce.",
                      f'{cfg["PROYECTO"]}.{cfg["DATASET_ORIGEN"]}.')
    sql = sql.replace("bussiness-case-tyc.tais_arrunategui.",
                      f'{cfg["PROYECTO"]}.{cfg["DATASET_DESTINO"]}.')
    return sql


def mostrar_sql(nombre):
    """Muestra el SQL dentro del notebook (queda visible en GitHub)."""
    display(Markdown(f"**`sql/{nombre}`**\n```sql\n{(DIR_SQL / nombre).read_text(encoding='utf-8')}\n```"))

## 1. Calidad de los datos fuente
Antes de construir nada se mide la calidad de las tablas que se usan (`users`, `orders`, `order_items`,
`products`). `distribution_centers` e `inventory_items` **no se usan**: no aportan información disponible
al momento de la primera compra que no esté ya en `products`.

In [ ]:
mostrar_sql("00a_calidad_fuentes.sql")

In [ ]:
def controlar_fuentes(cfg):
    """Controles de calidad de las fuentes. Devuelve la tabla de controles."""
    df = consultar(cargar_sql("00a_calidad_fuentes.sql", cfg))
    revisar = df[df.estado != "OK"]
    print(f"{len(df)} controles · {len(revisar)} para revisar (se tratan en la tabla base)")
    return df

calidad_fuentes = controlar_fuentes(CONFIG)
calidad_fuentes

In [ ]:
mostrar_sql("00b_estados_y_fechas.sql")
estados = consultar(cargar_sql("00b_estados_y_fechas.sql", CONFIG))
estados

**Cómo se trata cada hallazgo en la tabla base** (`01_tabla_base.sql`):

**Resultado en los datos reales:** los 16 controles dan OK (sin duplicados, nulos en claves, huérfanos ni
fechas futuras; historia de enero 2019 al 14/02/2025). Aun así, la tabla base aplica los tratamientos de
abajo de forma **preventiva**: el pipeline correrá sobre cargas futuras que sí pueden traer estos problemas.
El riesgo real de este caso no es la suciedad sino la **definición**: 15% de órdenes canceladas y 10% devueltas.

| Hallazgo | Tratamiento |
|---|---|
| Duplicados en `users`, `orders`, `order_items`, `products` | `QUALIFY ROW_NUMBER() ... = 1` por clave primaria |
| Nulos en claves (`order_id`, `user_id`, `created_at`) | Se excluyen: sin clave no hay cliente ni fecha |
| Órdenes sin usuario (huérfanas) | Se excluyen (`user_id IN usuarios`) |
| Ítems sin producto | Se conservan para el monto; categoría queda nula |
| Estado `Cancelled` | **Excluido**: no es una compra (parámetro `ESTADOS_EXCLUIDOS`) |
| Estado `Returned` | **Incluido**: hubo decisión de compra; la devolución ocurre después |
| `Processing` / `Shipped` / `Complete` | Incluidos: la orden fue realizada |
| Fechas futuras | Se descartan con `created_at < fecha_corte` |

## 2. Tabla base del modelo (SQL en BigQuery)

**Grano:** 1 fila por cliente. **Momento de predicción:** la primera compra.

**Decisiones clave**

| Decisión | Criterio |
|---|---|
| Primera compra | Primera orden **no cancelada** por cliente; desempate por `order_id` |
| Etiqueta | `recompra_60d = 1` si existe **otra orden** válida en (primera compra, primera compra + 60 días] |
| Clientes sin 60 días de historia | Primera compra en los últimos 60 días antes del corte → conjunto **scoring**, etiqueta `NULL`. No se entrenan con ellos (serían falsos negativos), pero **son justamente los clientes a contactar** |
| Joins sin duplicar | Ítems y eventos se **agregan a nivel orden/cliente antes** de unir |
| Eficiencia | Solo columnas necesarias; ítems filtrados a las primeras órdenes antes de agregar; sin `SELECT *` sobre fuentes |

**Prevención de fuga de información (leakage)**

| Variable descartada | Por qué sería fuga |
|---|---|
| `status` de la primera orden | Cambia después (p. ej. pasa a `Returned`) |
| `shipped_at`, `delivered_at`, `returned_at` | Ocurren después de la compra |
| Eventos web posteriores a la compra | Filtro `events.created_at < primera_compra_ts` |
| Nº total de órdenes o gasto histórico del cliente | Incluyen el futuro |
| Columnas `aux_*` | Solo auditoría y análisis; el modelo usa una **lista blanca** de variables |

In [ ]:
mostrar_sql("01_tabla_base.sql")

In [ ]:
def resolver_fecha_corte(cfg):
    """Fecha de corte explícita, o el 1er día del mes de la última orden observada.
    Es exclusiva: solo se usa información anterior a esa fecha."""
    if cfg["FECHA_CORTE"]:
        return cfg["FECHA_CORTE"]
    df = consultar(f"""
        SELECT DATE_TRUNC(DATE(MAX(created_at)), MONTH) AS fc
        FROM `{nombre_tabla(cfg, 'orders')}`
        WHERE created_at <= CURRENT_TIMESTAMP()""")
    return pd.Timestamp(df.fc.iloc[0]).date()


def parametros_sql(cfg, fecha_corte):
    return {"fecha_corte": fecha_corte,
            "ventana_dias": int(cfg["VENTANA_DIAS"]),
            "estados_excluidos": list(cfg["ESTADOS_EXCLUIDOS"])}


def construir_tabla_base(cfg, fecha_corte):
    """Ejecuta 01_tabla_base.sql con parámetros. Si GUARDAR_EN_BQ, la escribe en el dataset
    destino con WRITE_TRUNCATE (re-ejecutar no duplica). Devuelve el DataFrame."""
    destino = nombre_tabla(cfg, "base_recompra_60d", destino=True) if cfg["GUARDAR_EN_BQ"] else None
    if destino:
        asegurar_dataset_destino(cfg)
    df = consultar(cargar_sql("01_tabla_base.sql", cfg), parametros_sql(cfg, fecha_corte), destino=destino)
    return preparar_tipos(df)


def preparar_tipos(df):
    """Tipos homogéneos para pandas/sklearn (BigQuery devuelve enteros con nulos como Int64)."""
    df = df.copy()
    df["primera_compra_ts"] = pd.to_datetime(df["primera_compra_ts"], utc=True)
    for c in ("fecha_primera_compra", "fecha_corte"):
        df[c] = pd.to_datetime(df[c].astype(str)).dt.date
    for c in df.columns:
        if str(df[c].dtype) in ("Int64", "Float64", "Int32"):
            df[c] = df[c].astype("float64")
    return df

In [ ]:
mostrar_sql("02_validacion_tabla_base.sql")

In [ ]:
class ErrorCalidadDatos(Exception):
    """Se lanza cuando un control crítico falla: el pipeline se detiene."""


def validar_tabla_base(df, cfg, fecha_corte):
    """Validaciones en SQL (sobre la tabla escrita) y en Python (sobre el DataFrame)."""
    fallas = []
    if cfg["GUARDAR_EN_BQ"]:
        res = consultar(cargar_sql("02_validacion_tabla_base.sql", cfg))
        display(res)
        fallas += [f"SQL: {c}" for c in res.loc[res.estado == "FALLA", "control"]]

    obligatorias = {"user_id", "conjunto", "recompra_60d", "primera_compra_ts", *FEATURES}
    faltan = obligatorias - set(df.columns)
    if faltan:
        fallas.append(f"columnas faltantes: {sorted(faltan)}")
    if df.empty:
        fallas.append("tabla base vacía")
    if not df["user_id"].is_unique:
        fallas.append("user_id duplicado (algún join multiplicó filas)")
    ent = df[df.conjunto == "entrenamiento"]
    if ent["recompra_60d"].isna().any():
        fallas.append("etiqueta nula en entrenamiento")
    if not set(ent["recompra_60d"].dropna().unique()) <= {0, 1}:
        fallas.append("etiqueta con valores distintos de 0/1")
    if ent["recompra_60d"].nunique() < 2 or ent["recompra_60d"].sum() < 50:
        fallas.append("muy pocos positivos para entrenar")
    if (df["fecha_primera_compra"] >= fecha_corte).any():
        fallas.append("hay primeras compras en o después del corte")
    fuga = [c for c in FEATURES if c.startswith("aux_") or c == ETIQUETA]
    if fuga:
        fallas.append(f"columnas prohibidas en FEATURES: {fuga}")

    if fallas:
        raise ErrorCalidadDatos("Validación fallida:\n - " + "\n - ".join(fallas))
    nulos = df[FEATURES].isna().mean().loc[lambda s: s > 0].round(3)
    print(f"Tabla base válida: {len(df):,} clientes "
          f"({len(ent):,} entrenamiento · {(df.conjunto == 'scoring').sum():,} scoring)")
    if len(nulos):
        print("Proporción de nulos por variable (se imputan en el pipeline):", nulos.to_dict())

In [ ]:
# Lista blanca de variables del modelo: todas conocidas en el momento de la primera compra
ETIQUETA = "recompra_60d"
VARS_NUM = ["edad", "dias_registro_a_compra", "mes_primera_compra", "dia_semana_primera_compra",
            "n_items", "n_productos", "monto_total", "precio_max_item", "margen_bruto",
            "n_categorias", "n_marcas", "n_eventos_previos", "n_sesiones_previas",
            "n_vistas_producto", "n_eventos_carrito", "dias_primer_evento_a_compra"]
VARS_CAT = ["genero", "pais", "canal_adquisicion", "categoria_principal", "departamento_principal"]
FEATURES = VARS_NUM + VARS_CAT

FECHA_CORTE = resolver_fecha_corte(CONFIG)
print("Fecha de corte:", FECHA_CORTE)
base = construir_tabla_base(CONFIG, FECHA_CORTE)
validar_tabla_base(base, CONFIG, FECHA_CORTE)
base.head()

## 3. Análisis exploratorio
Solo lo que orienta el modelo y la acción comercial: cuántos recompran, **cuándo** recompran
(define el momento de la campaña) y **quiénes** recompran más.

In [ ]:
# Estilo de gráficos: superficie clara, ejes recesivos, una sola escala por gráfico
COLOR = {"serie1": "#2a78d6", "serie2": "#eb6834", "serie3": "#1baf7a",
         "texto": "#0b0b0b", "texto2": "#52514e", "grilla": "#e4e3df", "fondo": "#fcfcfb"}
plt.rcParams.update({
    "figure.facecolor": COLOR["fondo"], "axes.facecolor": COLOR["fondo"],
    "axes.edgecolor": COLOR["texto2"], "axes.labelcolor": COLOR["texto2"],
    "xtick.color": COLOR["texto2"], "ytick.color": COLOR["texto2"],
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": COLOR["grilla"], "grid.linewidth": 0.8,
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlecolor": COLOR["texto"],
    "font.size": 10, "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
})


def guardar_figura(fig, nombre):
    fig.savefig(DIR_FIGURAS / f"{nombre}.png")
    plt.show()


def pct(x):
    return f"{100 * x:.1f}%"

In [ ]:
ent = base[base.conjunto == "entrenamiento"].copy()
tasa_global = ent[ETIQUETA].mean()
ent["cohorte"] = pd.to_datetime(ent["fecha_primera_compra"]).dt.to_period("M").dt.to_timestamp()
por_cohorte = ent.groupby("cohorte")[ETIQUETA].agg(tasa="mean", clientes="size").reset_index()
por_cohorte = por_cohorte[por_cohorte.clientes >= 30]   # cohortes muy pequeñas son ruido

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(por_cohorte.cohorte, por_cohorte.tasa, color=COLOR["serie1"], lw=2, marker="o", ms=4)
ax.axhline(tasa_global, color=COLOR["texto2"], lw=1, ls="--")
ax.annotate(f"promedio {pct(tasa_global)}", (por_cohorte.cohorte.iloc[-1], tasa_global),
            xytext=(0, -16), textcoords="offset points", ha="right", color=COLOR["texto2"],
            bbox=dict(fc=COLOR["fondo"], ec="none", pad=1))
ax.set_title("Tasa de recompra a 60 días por cohorte de primera compra")
ax.set_ylabel("% de clientes que recompran")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{100*v:.0f}%"))
ax.set_ylim(0, max(por_cohorte.tasa.max() * 1.25, tasa_global * 1.5))
guardar_figura(fig, "fig1_tasa_recompra_cohorte")
print(f"Clientes con ventana completa: {len(ent):,} · recompran a 60 días: {pct(tasa_global)}")

In [ ]:
# ¿Cuándo ocurre la segunda orden? (entre quienes tienen una antes del corte)
dias = ent["aux_dias_a_segunda_orden"].dropna()
dentro = (dias <= CONFIG["VENTANA_DIAS"]).mean()
mediana_60 = dias[dias <= CONFIG["VENTANA_DIAS"]].median()

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(dias.clip(upper=365), bins=np.arange(0, 372, 7), color=COLOR["serie1"],
        edgecolor=COLOR["fondo"], linewidth=1)
ax.axvline(CONFIG["VENTANA_DIAS"], color=COLOR["serie2"], lw=2)
ax.annotate(f"ventana de {CONFIG['VENTANA_DIAS']} días\n{pct(dentro)} de las 2das órdenes",
            (CONFIG["VENTANA_DIAS"], ax.get_ylim()[1] * 0.85), xytext=(8, 0),
            textcoords="offset points", color=COLOR["texto"])
ax.set_title("Días entre la primera y la segunda orden")
ax.set_xlabel("días (365+ agrupado al final)")
ax.set_ylabel("clientes")
ax.grid(axis="x", visible=False)
guardar_figura(fig, "fig2_dias_a_segunda_orden")
print(f"De quienes recompran alguna vez, {pct(dentro)} lo hace dentro de {CONFIG['VENTANA_DIAS']} días; "
      f"dentro de la ventana, la mediana es {mediana_60:.0f} días → momento sugerido para la campaña.")

In [ ]:
def tasa_por(df, col, min_clientes=100, top=10):
    t = df.groupby(col)[ETIQUETA].agg(tasa="mean", clientes="size")
    return t[t.clientes >= min_clientes].sort_values("tasa").tail(top)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, col, titulo in [(axes[0], "canal_adquisicion", "Por canal de adquisición"),
                        (axes[1], "categoria_principal", "Por categoría de la 1ra compra (top 10)")]:
    t = tasa_por(ent, col)
    ax.barh(t.index.astype(str), t.tasa, color=COLOR["serie1"], height=0.6)
    ax.axvline(tasa_global, color=COLOR["texto2"], lw=1, ls="--")
    for y, (v, n) in enumerate(zip(t.tasa, t.clientes)):
        ax.text(v, y, f" {pct(v)}", va="center", color=COLOR["texto"], fontsize=9,
                bbox=dict(fc=COLOR["fondo"], ec="none", pad=0.5))
    ax.set_title(titulo, loc="left")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{100*v:.0f}%"))
    ax.set_xlim(0, t.tasa.max() * 1.25)
    ax.grid(axis="y", visible=False)
fig.suptitle("Tasa de recompra a 60 días por segmento (línea = promedio)", x=0.01, ha="left",
             fontweight="bold", color=COLOR["texto"])
fig.tight_layout()
guardar_figura(fig, "fig3_recompra_por_segmento")

## 4. Modelado y evaluación

**Separación de datos: temporal, no aleatoria.** Se entrena con las cohortes más antiguas y se evalúa con
el 20% más reciente, como ocurrirá en producción. Entre ambos conjuntos se deja un **hueco de 60 días**:
así ninguna etiqueta de entrenamiento se observa dentro del período de evaluación.

**Modelos comparados**
1. **Referencia de negocio:** tasa histórica de recompra del segmento *canal × departamento*
   (suavizada hacia el promedio). Es lo que el negocio haría sin modelo.
2. **Regresión logística:** lineal, interpretable.
3. **Gradient boosting (HistGradientBoosting):** captura no linealidades e interacciones; maneja nulos y categorías.

**Métricas:** la clase positiva es minoritaria, así que la accuracy engaña. Se usa **PR AUC** como métrica
principal (calidad del ranking sobre los que recompran) y, para la decisión de negocio, **precision, recall,
F1 y lift en el top `CAPACIDAD_CAMPANA`** de clientes, que es a quienes realmente se contactará.

In [ ]:
def split_temporal(df, fraccion_test, gap_dias):
    """Train = cohortes antiguas, test = más recientes, con hueco de `gap_dias` entre ambos."""
    df = df.sort_values(["primera_compra_ts", "user_id"]).reset_index(drop=True)
    inicio_test = df["primera_compra_ts"].quantile(1 - fraccion_test)
    fin_train = inicio_test - pd.Timedelta(days=gap_dias)
    train = df[df["primera_compra_ts"] < fin_train]
    test = df[df["primera_compra_ts"] >= inicio_test]
    assert train["primera_compra_ts"].max() < test["primera_compra_ts"].min()
    assert set(train.user_id).isdisjoint(test.user_id)
    return train, test


class ReferenciaSegmento(BaseEstimator, ClassifierMixin):
    """Referencia simple: tasa de recompra histórica del segmento, suavizada."""
    def __init__(self, columnas=("canal_adquisicion", "departamento_principal"), m=50):
        self.columnas, self.m = columnas, m

    def _clave(self, X):
        return X[list(self.columnas)].astype(str).agg("|".join, axis=1)

    def fit(self, X, y):
        y = pd.Series(np.asarray(y), index=X.index)
        self.global_ = y.mean()
        g = y.groupby(self._clave(X)).agg(["sum", "count"])
        self.tasas_ = ((g["sum"] + self.m * self.global_) / (g["count"] + self.m)).to_dict()
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        p = self._clave(X).map(self.tasas_).fillna(self.global_).to_numpy()
        return np.column_stack([1 - p, p])


SLUG_MODELO = {"Referencia (segmento)": "referencia", "Regresión logística": "logistica",
               "Gradient boosting": "hgb"}


def crear_modelos(semilla):
    imputar_cat = SimpleImputer(strategy="constant", fill_value="desconocido")
    logistica = Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), VARS_NUM),
            ("cat", Pipeline([("imp", imputar_cat),
                              ("ohe", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30))]), VARS_CAT),
        ])),
        ("modelo", LogisticRegression(max_iter=2000, C=0.5)),
    ])
    boosting = Pipeline([
        ("prep", ColumnTransformer([
            ("cat", Pipeline([("imp", imputar_cat),
                              ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]), VARS_CAT),
            ("num", "passthrough", VARS_NUM),     # HistGradientBoosting maneja nulos de forma nativa
        ])),
        ("modelo", HistGradientBoostingClassifier(
            categorical_features=list(range(len(VARS_CAT))), learning_rate=0.05, max_iter=400,
            max_leaf_nodes=15, min_samples_leaf=40, l2_regularization=1.0,
            early_stopping=True, validation_fraction=0.15, random_state=semilla)),
    ])
    return {"Referencia (segmento)": ReferenciaSegmento(),
            "Regresión logística": logistica,
            "Gradient boosting": boosting}


def metricas(y, p, capacidad):
    """Métricas de ranking + métricas en el punto de operación (top `capacidad` de clientes)."""
    y = np.asarray(y).astype(int)
    k = max(1, math.ceil(capacidad * len(y)))
    top = np.argsort(-p, kind="stable")[:k]
    tp = y[top].sum()
    prec, rec = tp / k, tp / max(1, y.sum())
    return {"pr_auc": average_precision_score(y, p), "roc_auc": roc_auc_score(y, p),
            "prevalencia": y.mean(), f"precision_top{int(capacidad*100)}": prec,
            f"recall_top{int(capacidad*100)}": rec,
            f"f1_top{int(capacidad*100)}": 2 * prec * rec / max(1e-9, prec + rec),
            f"lift_top{int(capacidad*100)}": prec / y.mean()}


def ic_bootstrap_pr_auc(y, p, n=300, semilla=42):
    """Intervalo de confianza 90% de la PR AUC por bootstrap del conjunto de test."""
    rng, y = np.random.default_rng(semilla), np.asarray(y).astype(int)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].sum() > 0:
            vals.append(average_precision_score(y[i], p[i]))
    return np.percentile(vals, [5, 95])


def entrenar_y_evaluar(ent, cfg):
    train, test = split_temporal(ent, cfg["FRACCION_TEST"], cfg["VENTANA_DIAS"])
    print(f"Train: {len(train):,} clientes ({train.fecha_primera_compra.min()} → {train.fecha_primera_compra.max()}) · "
          f"tasa {pct(train[ETIQUETA].mean())}")
    print(f"Test:  {len(test):,} clientes ({test.fecha_primera_compra.min()} → {test.fecha_primera_compra.max()}) · "
          f"tasa {pct(test[ETIQUETA].mean())}")
    modelos, filas, probs = crear_modelos(cfg["SEMILLA"]), [], {}
    for nombre, m in modelos.items():
        m.fit(train[FEATURES], train[ETIQUETA].astype(int))
        p = m.predict_proba(test[FEATURES])[:, 1]
        probs[nombre] = p
        lo, hi = ic_bootstrap_pr_auc(test[ETIQUETA], p, semilla=cfg["SEMILLA"])
        filas.append({"modelo": nombre, **metricas(test[ETIQUETA], p, cfg["CAPACIDAD_CAMPANA"]),
                      "pr_auc_ic90": f"[{lo:.3f}, {hi:.3f}]"})
    tabla = pd.DataFrame(filas).set_index("modelo")
    ganador = tabla["pr_auc"].idxmax()
    return {"train": train, "test": test, "modelos": modelos, "probs": probs,
            "tabla": tabla, "ganador": ganador}

In [ ]:
res = entrenar_y_evaluar(ent, CONFIG)
display(res["tabla"].round(3))
print("Modelo seleccionado por PR AUC en test:", res["ganador"])

In [ ]:
y_test = res["test"][ETIQUETA].astype(int).to_numpy()
fig, ax = plt.subplots(figsize=(6.5, 4.6))
colores = [COLOR["texto2"], COLOR["serie2"], COLOR["serie1"]]
for (nombre, p), c in zip(res["probs"].items(), colores):
    prec, rec, _ = precision_recall_curve(y_test, p)
    ax.plot(rec, prec, color=c, lw=2, label=f"{nombre} (PR AUC {average_precision_score(y_test, p):.3f})")
ax.axhline(y_test.mean(), color=COLOR["texto2"], lw=1, ls=":")
ax.annotate(f"azar = prevalencia {pct(y_test.mean())}", (0.02, y_test.mean()), xytext=(0, -14),
            textcoords="offset points", color=COLOR["texto2"])
ax.set_xlabel("Recall (recompradores encontrados)")
ax.set_ylabel("Precision (aciertos entre contactados)")
ax.set_title("Curva precision-recall en test (cohortes más recientes)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(frameon=False, loc="upper right")
guardar_figura(fig, "fig4_curva_pr")

In [ ]:
def tabla_deciles(y, p):
    """Decil 1 = clientes con mayor probabilidad."""
    d = pd.DataFrame({"y": np.asarray(y).astype(int), "p": p})
    d["decil"] = 10 - pd.qcut(d.p.rank(method="first"), 10, labels=False)
    t = d.groupby("decil").agg(clientes=("y", "size"), recompras=("y", "sum"), prob_media=("p", "mean"))
    t["tasa_real"] = t.recompras / t.clientes
    t["lift"] = t.tasa_real / d.y.mean()
    t["captura_acumulada"] = t.recompras.cumsum() / d.y.sum()
    return t

deciles = tabla_deciles(y_test, res["probs"][res["ganador"]])
display(deciles.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].bar(deciles.index, deciles.tasa_real, color=COLOR["serie1"], width=0.7)
axes[0].axhline(y_test.mean(), color=COLOR["texto2"], lw=1, ls="--")
axes[0].set_title("Tasa real de recompra por decil de score", loc="left")
axes[0].set_xlabel("decil (1 = mayor probabilidad)"); axes[0].set_xticks(deciles.index)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{100*v:.0f}%"))
axes[0].grid(axis="x", visible=False)
axes[1].plot(np.r_[0, deciles.index * 10], np.r_[0, deciles.captura_acumulada], color=COLOR["serie1"],
             lw=2, marker="o", ms=4, label="modelo")
axes[1].plot([0, 100], [0, 1], color=COLOR["texto2"], lw=1, ls=":", label="contacto al azar")
k = int(CONFIG["CAPACIDAD_CAMPANA"] * 100)
cap_k = deciles.captura_acumulada.iloc[k // 10 - 1]
axes[1].annotate(f"top {k}% → {pct(cap_k)}\nde los recompradores", (k, cap_k), xytext=(10, -28),
                 textcoords="offset points", color=COLOR["texto"])
axes[1].set_title("Recompradores capturados según % contactado", loc="left")
axes[1].set_xlabel("% de clientes contactados (ordenados por score)")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{100*v:.0f}%"))
axes[1].legend(frameon=False, loc="lower right")
fig.tight_layout()
guardar_figura(fig, "fig5_lift_deciles")

In [ ]:
# ¿Qué variables usa el modelo? Importancia por permutación sobre test (en caída de PR AUC)
imp = permutation_importance(res["modelos"][res["ganador"]], res["test"][FEATURES], y_test,
                             scoring="average_precision", n_repeats=5, random_state=CONFIG["SEMILLA"])
importancias = (pd.DataFrame({"variable": FEATURES, "caida_pr_auc": imp.importances_mean})
                .sort_values("caida_pr_auc", ascending=False).reset_index(drop=True))
importancias.head(10).round(4)

**Limitaciones**
- Datos **sintéticos**: la señal puede ser débil; el valor está en el ranking (lift), no en la probabilidad exacta.
- El modelo predice **quién recompra**, no **quién recompra gracias a la campaña**. Para eso: grupo de control
  y luego **modelo de uplift**.
- Variables solo del momento de la primera compra; no hay costos de campaña ni margen real por cliente.
- Una sola fecha de corte evaluada; en producción se haría *backtesting* con varias.

**Mejoras**
- *Backtesting* mensual con varias fechas de corte y monitoreo de PR AUC por cohorte.
- Calibración de probabilidades (isotónica) si se usan para cálculos de valor esperado.
- Variables de navegación más ricas (recencia de sesiones, productos vistos y no comprados).
- Umbral de contacto según costo de campaña y margen esperado.

## 5. Predicciones para el negocio
Se reentrena el modelo seleccionado con **todos** los clientes etiquetados y se puntúan los clientes del
conjunto **scoring** (primera compra en los últimos 60 días, ventana aún abierta). Son los clientes sobre
los que todavía se puede actuar.

In [ ]:
def entrenar_final(ent, nombre_modelo, cfg):
    modelo = crear_modelos(cfg["SEMILLA"])[nombre_modelo]
    modelo.fit(ent[FEATURES], ent[ETIQUETA].astype(int))
    return modelo


def predecir(modelo, df_scoring, fecha_corte, cfg, version):
    """Probabilidad, decil y acción sugerida por cliente."""
    p = modelo.predict_proba(df_scoring[FEATURES])[:, 1]
    out = df_scoring[["user_id", "fecha_primera_compra", "aux_dias_observados",
                      "aux_ya_recompro_al_corte"]].copy()
    out = out.rename(columns={"aux_dias_observados": "dias_desde_primera_compra",
                              "aux_ya_recompro_al_corte": "ya_recompro"})
    out["probabilidad_recompra"] = p.round(5)
    out["decil"] = (10 - pd.qcut(pd.Series(p).rank(method="first"), 10, labels=False)).to_numpy() if len(p) >= 10 else 1
    corte_prioridad = np.quantile(p, 1 - cfg["CAPACIDAD_CAMPANA"]) if len(p) else 1
    out["accion"] = np.select(
        [out.ya_recompro == 1, p >= corte_prioridad],
        ["Ya recompró: no contactar", "Prioridad alta: contactar"], "Sin contacto / canal de bajo costo")
    out["fecha_corte"] = fecha_corte
    out["modelo_version"] = version
    out["fecha_ejecucion"] = pd.Timestamp.now(tz="UTC")
    return out.sort_values("probabilidad_recompra", ascending=False).reset_index(drop=True)


def validar_predicciones(pred, df_scoring):
    errores = []
    if not pred.user_id.is_unique: errores.append("user_id duplicado en predicciones")
    if len(pred) != len(df_scoring): errores.append("no se puntuaron todos los clientes")
    if not pred.probabilidad_recompra.between(0, 1).all(): errores.append("probabilidad fuera de [0, 1]")
    if errores:
        raise ErrorCalidadDatos(" | ".join(errores))


def guardar_resultados(modelo, pred, metr, cfg, fecha_corte, version):
    """Escrituras idempotentes: archivos con nombre por fecha de corte (se sobrescriben)
    y partición por fecha de corte en BigQuery (se reemplaza)."""
    sufijo = fecha_corte.strftime("%Y%m%d")
    joblib.dump({"modelo": modelo, "features": FEATURES, "version": version,
                 "fecha_corte": fecha_corte, "config": cfg, "metricas_test": metr},
                DIR_MODELOS / "modelo_recompra_60d.joblib")
    pred.to_csv(DIR_SALIDAS / f"predicciones_{sufijo}.csv", index=False)
    pred[["user_id", "probabilidad_recompra", "decil", "accion"]].head(500).to_csv(
        DIR_SALIDAS / "predicciones_muestra.csv", index=False)
    (DIR_SALIDAS / "metricas.json").write_text(json.dumps(metr, indent=2, default=str), encoding="utf-8")
    if cfg["GUARDAR_EN_BQ"]:
        tabla = nombre_tabla(cfg, "predicciones_recompra_60d", destino=True)
        reemplazar_particion(pred, tabla, fecha_corte)
        n = contar_filas(tabla, fecha_corte)
        assert n == len(pred), f"BigQuery tiene {n} filas para {fecha_corte}, se esperaban {len(pred)}"
        print(f"Predicciones en {tabla} (partición {fecha_corte}): {n:,} filas")

In [ ]:
scoring = base[base.conjunto == "scoring"]
VERSION = f'{SLUG_MODELO[res["ganador"]]}_{FECHA_CORTE:%Y%m%d}'
modelo_final = entrenar_final(ent, res["ganador"], CONFIG)
pred = predecir(modelo_final, scoring, FECHA_CORTE, CONFIG, VERSION)
validar_predicciones(pred, scoring)
metr = {"fecha_corte": FECHA_CORTE, "modelo": res["ganador"], "version": VERSION,
        "tasa_recompra_entrenamiento": tasa_global,
        "test": res["tabla"].drop(columns="pr_auc_ic90").round(4).to_dict(orient="index"),
        "pr_auc_ic90": res["tabla"]["pr_auc_ic90"].to_dict(),
        "captura_top_capacidad": float(cap_k), "clientes_scoring": len(pred)}
guardar_resultados(modelo_final, pred, metr, CONFIG, FECHA_CORTE, VERSION)
display(pred.accion.value_counts())
pred.head(10)

## 6. Ejecución de punta a punta
`ejecutar_pipeline(cfg)` encadena todo sin pasos manuales: fecha de corte → tabla base → validaciones →
entrenamiento y evaluación → predicciones → guardado. Cambiar `FECHA_CORTE` o `VENTANA_DIAS` en `CONFIG`
basta para re-entrenar con otro corte.

Para comprobar que **re-ejecutar no duplica resultados**, se corre el pipeline completo y se verifica que la
cantidad de predicciones guardadas para la fecha de corte coincide con la de la ejecución anterior.

In [ ]:
def ejecutar_pipeline(cfg):
    fecha_corte = resolver_fecha_corte(cfg)
    print(f"▶ Fecha de corte {fecha_corte} · ventana {cfg['VENTANA_DIAS']} días")
    df = construir_tabla_base(cfg, fecha_corte)
    validar_tabla_base(df, cfg, fecha_corte)
    etiquetados = df[df.conjunto == "entrenamiento"]
    r = entrenar_y_evaluar(etiquetados, cfg)
    version = f'{SLUG_MODELO[r["ganador"]]}_{fecha_corte:%Y%m%d}'
    modelo = entrenar_final(etiquetados, r["ganador"], cfg)
    scoring = df[df.conjunto == "scoring"]
    pred = predecir(modelo, scoring, fecha_corte, cfg, version)
    validar_predicciones(pred, scoring)
    metr = {"fecha_corte": fecha_corte, "modelo": r["ganador"], "version": version,
            "tasa_recompra_entrenamiento": etiquetados[ETIQUETA].mean(),
            "test": r["tabla"].drop(columns="pr_auc_ic90").round(4).to_dict(orient="index"),
            "pr_auc_ic90": r["tabla"]["pr_auc_ic90"].to_dict(),
            "captura_top_capacidad": float(tabla_deciles(r["test"][ETIQUETA], r["probs"][r["ganador"]])
                                           .captura_acumulada.iloc[int(cfg["CAPACIDAD_CAMPANA"] * 10) - 1]),
            "clientes_scoring": len(pred)}
    guardar_resultados(modelo, pred, metr, cfg, fecha_corte, version)
    print(f"✔ Pipeline completo: {len(pred):,} clientes puntuados con {r['ganador']}")
    return pred, metr

pred_2, metr_2 = ejecutar_pipeline(CONFIG)

# Prueba de idempotencia: misma fecha de corte → mismos clientes, sin filas duplicadas
assert len(pred_2) == len(pred) and set(pred_2.user_id) == set(pred.user_id)
csv = pd.read_csv(DIR_SALIDAS / f"predicciones_{FECHA_CORTE:%Y%m%d}.csv")
assert csv.user_id.is_unique and len(csv) == len(pred)
print("✔ Re-ejecución idempotente: no se duplicaron resultados")

## Escalabilidad (comentario)
**BigQuery con más volumen:** particionar `orders`, `order_items` y `events` por `DATE(created_at)` y
agruparlos (*clustering*) por `user_id`; procesar solo las cohortes nuevas de forma incremental en lugar de
reconstruir toda la tabla; leer solo columnas necesarias (ya aplicado); y agregar ítems y eventos antes de unir
(ya aplicado), que es lo que evita los joins explosivos.

**En Snowflake:** el mismo SQL con cambios menores (`QUALIFY` existe; `COUNTIF` → `COUNT_IF`;
`TIMESTAMP_ADD` → `DATEADD`). *Clustering keys* por fecha en las tablas grandes, un *warehouse* dedicado
al proceso de ML que se suspende al terminar, carga incremental con **Streams + Tasks**, y la tabla de
predicciones expuesta a negocio mediante una vista con permisos por rol. El detalle está en
`docs/arquitectura.md`.